In [3]:
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from pathlib import Path

In [4]:
import yfinance as yf

In [5]:
# 1. Configuration de l'univers

COMMODITIES = {
    "WTI": {
        "name": "Pétrole WTI",
        "ticker": "CL=F",
        "category": "Énergie",
        "priority": "A",
        "rss_query": '"WTI crude oil" OR "US crude oil" OR OPEC OR "oil inventories"'
    },
    "GOLD": {
        "name": "Or",
        "ticker": "GC=F",
        "category": "Métaux précieux",
        "priority": "A",
        "rss_query": '"gold price" OR bullion OR "central bank gold" OR "real yields"'
    },
    "NATURAL_GAS": {
        "name": "Gaz naturel Henry Hub",
        "ticker": "NG=F",
        "category": "Énergie",
        "priority": "A",
        "rss_query": '"Henry Hub" OR "natural gas storage" OR LNG OR "gas inventories"'
    },
    "COPPER": {
        "name": "Cuivre",
        "ticker": "HG=F",
        "category": "Métaux industriels",
        "priority": "A",
        "rss_query": '"copper price" OR "copper inventories" OR "copper mine" OR "China copper"'
    },
    "WHEAT": {
        "name": "Blé Chicago SRW",
        "ticker": "ZW=F",
        "category": "Agriculture",
        "priority": "A",
        "rss_query": '"wheat prices" OR "wheat harvest" OR "Black Sea wheat" OR "wheat exports"'
    },
    "CORN": {
        "name": "Maïs",
        "ticker": "ZC=F",
        "category": "Agriculture",
        "priority": "A",
        "rss_query": '"corn prices" OR "corn harvest" OR "corn crop" OR ethanol USDA'
    },
    "COCOA": {
        "name": "Cacao",
        "ticker": "CC=F",
        "category": "Soft commodities",
        "priority": "A",
        "rss_query": '"cocoa prices" OR "cocoa crop" OR "Ivory Coast cocoa" OR "Ghana cocoa"'
    },
    "COFFEE": {
        "name": "Café Arabica",
        "ticker": "KC=F",
        "category": "Soft commodities",
        "priority": "A",
        "rss_query": '"arabica coffee" OR "coffee prices" OR "Brazil coffee crop" OR "coffee frost"'
    },
    "BRENT": {
        "name": "Pétrole Brent",
        "ticker": "BZ=F",
        "category": "Énergie",
        "priority": "B",
        "rss_query": '"Brent crude" OR "Brent oil" OR OPEC OR "global oil supply"'
    },
    "SILVER": {
        "name": "Argent",
        "ticker": "SI=F",
        "category": "Métaux précieux",
        "priority": "B",
        "rss_query": '"silver price" OR "silver demand" OR "silver market" OR "solar silver"'
    },
    "SOYBEAN": {
        "name": "Soja",
        "ticker": "ZS=F",
        "category": "Agriculture",
        "priority": "B",
        "rss_query": '"soybean prices" OR "soybean crop" OR "Brazil soybeans" OR "China soybean imports"'
    },
    "SUGAR": {
        "name": "Sucre No. 11",
        "ticker": "SB=F",
        "category": "Soft commodities",
        "priority": "B",
        "rss_query": '"sugar prices" OR "Brazil sugar" OR "India sugar exports" OR "sugar crop"'
    },
    "PLATINUM": {
        "name": "Platine",
        "ticker": "PL=F",
        "category": "Métaux précieux",
        "priority": "C",
        "rss_query": '"platinum price" OR "platinum supply" OR "South Africa platinum" OR hydrogen'
    },
    "PALLADIUM": {
        "name": "Palladium",
        "ticker": "PA=F",
        "category": "Métaux précieux",
        "priority": "C",
        "rss_query": '"palladium price" OR "palladium supply" OR "Russia palladium" OR autocatalyst'
    },
    "COTTON": {
        "name": "Coton",
        "ticker": "CT=F",
        "category": "Agriculture",
        "priority": "C",
        "rss_query": '"cotton prices" OR "cotton crop" OR "cotton harvest" OR "China cotton"'
    }
}

# 2. Paramètres de téléchargement

START_DATE = "2015-01-01"
END_DATE = None          
INTERVAL = "1d"

# Pour ne récupérer que les marchés prioritaires :
# Pour récupérer les 15 marchés :
SELECTED_PRIORITIES = {"A", "B", "C"}

selected_commodities = {
    key: value
    for key, value in COMMODITIES.items()
    if value["priority"] in SELECTED_PRIORITIES
}

# 3. Téléchargement des prix

price_frames = []
download_errors = []

for commodity_id, config in selected_commodities.items():
    try:
        data = yf.download(
            config["ticker"],
            start=START_DATE,
            end=END_DATE,
            interval=INTERVAL,
            auto_adjust=False,
            progress=False,
            threads=False
        )

        if data.empty:
            download_errors.append({
                "commodity_id": commodity_id,
                "ticker": config["ticker"],
                "error": "Aucune donnée retournée"
            })
            continue

        # Certains retours yfinance utilisent des colonnes MultiIndex
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        data = data.reset_index()

        # Uniformisation du nom de la colonne temporelle
        date_column = "Datetime" if "Datetime" in data.columns else "Date"
        data = data.rename(columns={date_column: "date"})

        data["commodity_id"] = commodity_id
        data["commodity_name"] = config["name"]
        data["ticker"] = config["ticker"]
        data["category"] = config["category"]
        data["priority"] = config["priority"]

        data.columns = [column.lower().replace(" ", "_") for column in data.columns]

        expected_columns = [
            "date",
            "commodity_id",
            "commodity_name",
            "ticker",
            "category",
            "priority",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume"
        ]

        available_columns = [
            column for column in expected_columns
            if column in data.columns
        ]

        price_frames.append(data[available_columns])

    except Exception as error:
        download_errors.append({
            "commodity_id": commodity_id,
            "ticker": config["ticker"],
            "error": str(error)
        })

prices_df = (
    pd.concat(price_frames, ignore_index=True)
    if price_frames
    else pd.DataFrame()
)

if not prices_df.empty:
    prices_df["date"] = pd.to_datetime(
        prices_df["date"],
        utc=True,
        errors="coerce"
    )

    prices_df = (
        prices_df
        .drop_duplicates(subset=["date", "commodity_id"])
        .sort_values(["commodity_id", "date"])
        .reset_index(drop=True)
    )

# 4. Table de référence des matières

commodities_df = pd.DataFrame([
    {
        "commodity_id": commodity_id,
        "commodity_name": config["name"],
        "ticker": config["ticker"],
        "category": config["category"],
        "priority": config["priority"],
        "rss_query": config["rss_query"],
        "rss_url": (
            "https://news.google.com/rss/search?"
            f"q={quote_plus(config['rss_query'])}"
            "&hl=en-US&gl=US&ceid=US:en"
        )
    }
    for commodity_id, config in selected_commodities.items()
])

# 5. Contrôles

print(f"Nombre de matières premières : {len(commodities_df)}")
print(f"Nombre de lignes de prix : {len(prices_df):,}")
print(f"Période : {prices_df['date'].min()} → {prices_df['date'].max()}")

if download_errors:
    print("\nErreurs rencontrées :")
    display(pd.DataFrame(download_errors))

display(commodities_df)
display(prices_df.head())

Nombre de matières premières : 15
Nombre de lignes de prix : 43,295
Période : 2015-01-02 00:00:00+00:00 → 2026-06-26 00:00:00+00:00


,commodity_id,commodity_name,ticker,category,priority,rss_query,rss_url
0,WTI,Pétrole WTI,CL=F,Énergie,A,"""WTI crude oil"" OR ""US crude oil"" OR OPEC OR ""...",https://news.google.com/rss/search?q=%22WTI+cr...
1,GOLD,Or,GC=F,Métaux précieux,A,"""gold price"" OR bullion OR ""central bank gold""...",https://news.google.com/rss/search?q=%22gold+p...
2,NATURAL_GAS,Gaz naturel Henry Hub,NG=F,Énergie,A,"""Henry Hub"" OR ""natural gas storage"" OR LNG OR...",https://news.google.com/rss/search?q=%22Henry+...
3,COPPER,Cuivre,HG=F,Métaux industriels,A,"""copper price"" OR ""copper inventories"" OR ""cop...",https://news.google.com/rss/search?q=%22copper...
4,WHEAT,Blé Chicago SRW,ZW=F,Agriculture,A,"""wheat prices"" OR ""wheat harvest"" OR ""Black Se...",https://news.google.com/rss/search?q=%22wheat+...
5,CORN,Maïs,ZC=F,Agriculture,A,"""corn prices"" OR ""corn harvest"" OR ""corn crop""...",https://news.google.com/rss/search?q=%22corn+p...
6,COCOA,Cacao,CC=F,Soft commodities,A,"""cocoa prices"" OR ""cocoa crop"" OR ""Ivory Coast...",https://news.google.com/rss/search?q=%22cocoa+...
7,COFFEE,Café Arabica,KC=F,Soft commodities,A,"""arabica coffee"" OR ""coffee prices"" OR ""Brazil...",https://news.google.com/rss/search?q=%22arabic...
8,BRENT,Pétrole Brent,BZ=F,Énergie,B,"""Brent crude"" OR ""Brent oil"" OR OPEC OR ""globa...",https://news.google.com/rss/search?q=%22Brent+...
9,SILVER,Argent,SI=F,Métaux précieux,B,"""silver price"" OR ""silver demand"" OR ""silver m...",https://news.google.com/rss/search?q=%22silver...


,date,commodity_id,commodity_name,ticker,category,priority,open,high,low,close,adj_close,volume
0,2015-01-02 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,57.630001,58.220001,55.520000,56.419998,56.419998,16707
1,2015-01-05 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,56.290001,56.290001,52.669998,53.110001,53.110001,30065
2,2015-01-06 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,53.230000,53.520000,50.529999,51.099998,51.099998,35494
3,2015-01-07 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,51.060001,51.840000,49.680000,51.150002,51.150002,37082
4,2015-01-08 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,51.000000,51.889999,49.820000,50.959999,50.959999,29469


In [6]:
# NETTOYAGE : DOUBLONS ET VALEURS MANQUANTES
# Nombre maximal de lignes consécutives à interpoler
# Une limite faible évite de fabriquer artificiellement de longues périodes.
MAX_INTERPOLATION_GAP = 2

prices_clean_df = prices_df.copy()

# 1. Uniformisation des types

prices_clean_df["date"] = pd.to_datetime(
    prices_clean_df["date"],
    utc=True,
    errors="coerce"
)

numeric_columns = [
    col for col in [
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]
    if col in prices_clean_df.columns
]

for col in numeric_columns:
    prices_clean_df[col] = pd.to_numeric(
        prices_clean_df[col],
        errors="coerce"
    )

# Remplacement des valeurs infinies par NaN
prices_clean_df[numeric_columns] = prices_clean_df[
    numeric_columns
].replace([float("inf"), float("-inf")], pd.NA)

# Suppression des lignes sans date ou sans identifiant
prices_clean_df = prices_clean_df.dropna(
    subset=["date", "commodity_id"]
)

# 2. Suppression des doublons

rows_before = len(prices_clean_df)

# Suppression des doublons parfaitement identiques
prices_clean_df = prices_clean_df.drop_duplicates()

# Pour plusieurs lignes ayant la même matière et la même date,
# on conserve celle contenant le plus de valeurs renseignées.
prices_clean_df["_completeness"] = (
    prices_clean_df[numeric_columns]
    .notna()
    .sum(axis=1)
)

prices_clean_df = (
    prices_clean_df
    .sort_values(
        ["commodity_id", "date", "_completeness"],
        ascending=[True, True, False]
    )
    .drop_duplicates(
        subset=["commodity_id", "date"],
        keep="first"
    )
    .drop(columns="_completeness")
)

duplicates_removed = rows_before - len(prices_clean_df)

# 3. Rapport des valeurs manquantes avant traitement

missing_before = prices_clean_df[numeric_columns].isna().sum()

# Création d'indicateurs pour conserver la traçabilité
for col in numeric_columns:
    prices_clean_df[f"{col}_was_missing"] = (
        prices_clean_df[col].isna().astype("int8")
    )

# 4. Traitement des valeurs manquantes

prices_clean_df = prices_clean_df.sort_values(
    ["commodity_id", "date"]
).reset_index(drop=True)

price_columns = [
    col for col in ["open", "high", "low", "close"]
    if col in prices_clean_df.columns
]

# Interpolation limitée à l'intérieur des séries.
# Les valeurs au début et à la fin ne sont pas extrapolées.
for col in price_columns:
    prices_clean_df[col] = (
        prices_clean_df
        .groupby("commodity_id")[col]
        .transform(
            lambda series: series.interpolate(
                method="linear",
                limit=MAX_INTERPOLATION_GAP,
                limit_area="inside"
            )
        )
    )

# Si la clôture existe, elle peut servir à compléter un OHLC manquant
if "close" in prices_clean_df.columns:
    for col in ["open", "high", "low"]:
        if col in prices_clean_df.columns:
            prices_clean_df[col] = prices_clean_df[col].fillna(
                prices_clean_df["close"]
            )

# Yahoo Finance fournit parfois Adj Close vide pour certains futures
if {"adj_close", "close"}.issubset(prices_clean_df.columns):
    prices_clean_df["adj_close"] = prices_clean_df[
        "adj_close"
    ].fillna(prices_clean_df["close"])

# Le volume n'est pas interpolé, car cela créerait un faux volume.
# On conserve NaN et on ajoute une version exploitable par les modèles.
if "volume" in prices_clean_df.columns:
    prices_clean_df["volume_filled"] = (
        prices_clean_df["volume"].fillna(0)
    )

# 5. Suppression des lignes inutilisables

rows_before_close_filter = len(prices_clean_df)

# Une ligne sans clôture ne permet pas de calculer les rendements
prices_clean_df = prices_clean_df.dropna(subset=["close"])

rows_without_close_removed = (
    rows_before_close_filter - len(prices_clean_df)
)

# 6. Contrôle de cohérence OHLC

required_ohlc = {"open", "high", "low", "close"}

if required_ohlc.issubset(prices_clean_df.columns):
    prices_clean_df["high"] = prices_clean_df[
        ["open", "high", "low", "close"]
    ].max(axis=1)

    prices_clean_df["low"] = prices_clean_df[
        ["open", "high", "low", "close"]
    ].min(axis=1)

# Les prix négatifs ou nuls sont considérés comme invalides.
# Attention : le pétrole WTI a historiquement connu des prix négatifs.
# On ne filtre donc pas automatiquement CL=F.
price_check_columns = [
    col for col in ["open", "high", "low", "close"]
    if col in prices_clean_df.columns
]

invalid_price_mask = (
    prices_clean_df[price_check_columns].le(0).any(axis=1)
    & prices_clean_df["ticker"].ne("CL=F")
)

invalid_prices_removed = int(invalid_price_mask.sum())

prices_clean_df = prices_clean_df.loc[
    ~invalid_price_mask
].copy()

# 7. Rapport final

missing_after = prices_clean_df[numeric_columns].isna().sum()

missing_report = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after,
})

missing_report["values_treated"] = (
    missing_report["missing_before"]
    - missing_report["missing_after"]
)

missing_report["missing_rate_after_pct"] = (
    missing_report["missing_after"]
    / len(prices_clean_df)
    * 100
).round(2)

prices_clean_df = (
    prices_clean_df
    .sort_values(["commodity_id", "date"])
    .reset_index(drop=True)
)

print("Nettoyage terminé ✅")
print(f"Doublons supprimés : {duplicates_removed:,}")
print(
    f"Lignes supprimées car Close manquant : "
    f"{rows_without_close_removed:,}"
)
print(
    f"Lignes supprimées pour prix invalides : "
    f"{invalid_prices_removed:,}"
)
print(f"Lignes finales : {len(prices_clean_df):,}")

display(missing_report)
display(prices_clean_df.head())

Nettoyage terminé ✅
Doublons supprimés : 0
Lignes supprimées car Close manquant : 0
Lignes supprimées pour prix invalides : 0
Lignes finales : 43,295


,missing_before,missing_after,values_treated,missing_rate_after_pct
open,0,0,0,0.0
high,0,0,0,0.0
low,0,0,0,0.0
close,0,0,0,0.0
adj_close,0,0,0,0.0
volume,0,0,0,0.0


,date,commodity_id,commodity_name,ticker,category,priority,open,high,low,close,adj_close,volume,open_was_missing,high_was_missing,low_was_missing,close_was_missing,adj_close_was_missing,volume_was_missing,volume_filled
0,2015-01-02 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,57.630001,58.220001,55.520000,56.419998,56.419998,16707,0,0,0,0,0,0,16707
1,2015-01-05 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,56.290001,56.290001,52.669998,53.110001,53.110001,30065,0,0,0,0,0,0,30065
2,2015-01-06 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,53.230000,53.520000,50.529999,51.099998,51.099998,35494,0,0,0,0,0,0,35494
3,2015-01-07 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,51.060001,51.840000,49.680000,51.150002,51.150002,37082,0,0,0,0,0,0,37082
4,2015-01-08 00:00:00+00:00,BRENT,Pétrole Brent,BZ=F,Énergie,B,51.000000,51.889999,49.820000,50.959999,50.959999,29469,0,0,0,0,0,0,29469


In [8]:
OUTPUT_FOLDER = Path("data/raw/market_data")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_FOLDER / "data_raw_clean.csv"

prices_clean_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

print(f"Dataset sauvegardé dans : {OUTPUT_PATH.resolve()} ✅")

Dataset sauvegardé dans : /Users/thomas/Documents/GitHub/Projet-outil-ETL/data/raw/market_data/data_raw_clean.csv ✅
